# eXueed — Retrieval Backbone + Router + Safety Test

This notebook exercises the **new consolidated pipeline** introduced on
`claude/pipeline-consolidation`:

1. `retrieve_evidence(mode=...)` — single entry point returning an `EvidenceBundle` (Phase 2).
2. `pick_surface(extracted_axes, query_mode, force_trial_match)` — router that picks P1/P3/P4/P5 (Phase 3).
3. `safety.numerical` + `safety.citations` — post-generation strip passes (Phase 1b, Phase 5).
4. `pipeline_metrics` — per-request counter collector emitting a `[PipelineMetrics] …` summary line.

For each of three similar-detail patient cases the notebook:

- runs `pick_surface` on the extracted axes and prints the chosen surface
- runs `retrieve_evidence(mode="comprehensive")` and captures the bundle
- scores each retrieved study and prints **what criteria matched** between
  the query's extracted axes and each document's metadata / chunk text
- prints the active `PipelineMetrics` summary line for that run

At the end, a side-by-side comparison table shows how the three cases ranked
on matching-criteria coverage and raw scores.

> **Prereqs:** Qdrant, OpenAI, and (optional) Postgres credentials. Branch
> `claude/pipeline-consolidation` checked out.

## 1. Get the repo on the machine and install deps

In [ ]:
import os, sys, subprocess, getpass

REPO_OWNER = "alexandrahalfon"
REPO_NAME  = "exueed-updated"
BRANCH     = "claude/pipeline-consolidation"

ON_COLAB = "google.colab" in sys.modules
REPO_DIR = f"/content/{REPO_NAME}" if ON_COLAB else os.path.abspath("..")

already_present = os.path.isdir(os.path.join(REPO_DIR, "src", "api"))

if ON_COLAB and not already_present:
    token = os.environ.get("GITHUB_TOKEN")
    if not token:
        try:
            from google.colab import userdata  # type: ignore
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
    if not token:
        token = getpass.getpass("GitHub PAT (leave blank if repo already uploaded): ").strip()
    if not token:
        raise RuntimeError(f"No repo at {REPO_DIR} and no GITHUB_TOKEN given")
    url = f"https://{token}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
    subprocess.check_call(["git", "clone", "--branch", BRANCH, "--depth", "1", url, REPO_DIR])

if ON_COLAB:
    subprocess.check_call(["pip", "install", "-q",
        "qdrant-client", "openai", "python-dotenv", "pydantic", "pydantic-settings",
        "fastapi", "asyncpg", "psycopg2-binary", "sentence-transformers",
        "numpy", "scikit-learn", "tiktoken", "rapidfuzz", "httpx",
    ])

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print("Repo:", REPO_DIR, "| branch:", BRANCH)

## 2. Credentials

Set these **before** importing any service module — the settings singleton reads `os.environ` at import time.

In [ ]:
import os

os.environ["OPENAI_API_KEY"]    = os.environ.get("OPENAI_API_KEY",    "sk-...")
os.environ["QDRANT_URL"]        = os.environ.get("QDRANT_URL",        "https://<your-qdrant>.cloud.qdrant.io")
os.environ["QDRANT_API_KEY"]    = os.environ.get("QDRANT_API_KEY",    "...")
os.environ["QDRANT_COLLECTION"] = os.environ.get("QDRANT_COLLECTION", "exueed_kb_latest")

os.environ.setdefault("POSTGRES_HOST",     "34.21.60.224")
os.environ.setdefault("POSTGRES_PORT",     "5432")
os.environ.setdefault("POSTGRES_USER",     "postgres")
os.environ.setdefault("POSTGRES_PASSWORD", "")
os.environ.setdefault("POSTGRES_DATABASE", "display-study-details")

assert os.environ["OPENAI_API_KEY"].startswith("sk-"), "Set OPENAI_API_KEY"
assert os.environ["QDRANT_URL"].startswith("http"), "Set QDRANT_URL"
print("Credentials OK. Collection:", os.environ["QDRANT_COLLECTION"])

## 3. Ensure Qdrant payload indexes (one-time, idempotent)

Guarantees the typed `MatchAny` filters hit indexes instead of scanning.

In [ ]:
from src.ingestion.qdrant_client import QdrantIngestionClient

QdrantIngestionClient().ensure_collection()

## 4. Test queries

Three similar-detail patient narratives — same sentence budget (~90–110 words),
same clinical structure (age, sex, PMH, primary, prior therapy, current status,
question). The narratives span different tumour sites so recall behaviour can
be compared across categories.

Each query carries a `expected_axes` dict that is **not** used by the pipeline
— it is only the test fixture's answer key for the matching-criteria report.

In [ ]:
TEST_QUERIES = [
    {
        "id": "hn_ici_refractory",
        "narrative": (
            "80 y.o. male non-smoker, PMH HTN, Hep C, BPH, CKD, with recurrent SCC of the "
            "left oral tongue (CPS 100), s/p left partial glossectomy and neck dissection, "
            "progressing on pembrolizumab, no longer a surgical candidate, with radiographic "
            "concern for right-ventricle metastasis. What is the best next-line systemic therapy?"
        ),
        "expected_axes": {
            "category": "head_neck",
            "site": "oral tongue",
            "histology": "squamous cell carcinoma",
            "biomarkers": ["CPS 100", "PD-L1"],
            "trajectory": ["ici_refractory", "progressing_on_ici"],
            "surgical_candidate": False,
        },
    },
    {
        "id": "lung_egfr_osi",
        "narrative": (
            "72 y.o. female never-smoker, PMH HTN, mild CKD, with recurrent EGFR-mutant "
            "(exon 19 del) adenocarcinoma of the right lower lobe, s/p lobectomy and adjuvant "
            "osimertinib, now progressing on osimertinib with new liver metastases and "
            "radiographic concern for leptomeningeal spread. What is the best next-line "
            "systemic therapy?"
        ),
        "expected_axes": {
            "category": "lung",
            "site": "lung",
            "histology": "adenocarcinoma",
            "biomarkers": ["EGFR exon 19 del"],
            "trajectory": ["progression", "tki_refractory"],
            "surgical_candidate": False,
        },
    },
    {
        "id": "prostate_mcrpc",
        "narrative": (
            "68 y.o. male, PMH HTN, hyperlipidemia, T2DM, with metastatic castration-resistant "
            "prostate adenocarcinoma, Gleason 9, BRCA2-mutated, s/p ADT and abiraterone, now "
            "progressing on enzalutamide with new bone and nodal metastases and a rising PSA. "
            "What is the best next-line systemic therapy?"
        ),
        "expected_axes": {
            "category": "prostate",
            "site": "prostate",
            "histology": "adenocarcinoma",
            "biomarkers": ["BRCA2"],
            "trajectory": ["progression", "mCRPC"],
            "surgical_candidate": False,
        },
    },
]

for q in TEST_QUERIES:
    print(f"[{q['id']}] {q['narrative'][:100]}…")

## 5. Matching-criteria analyzer

For each returned study we compute a **criteria vector** against the query's
extracted axes. Each criterion that matched contributes to a composite
`criteria_score` (0.0 – 1.0). This is *in addition to* the retrieval/
rerank score the pipeline itself assigned.

| Criterion | How it's detected |
|---|---|
| `category` | `query.filter_category == study.category` |
| `site` | Query cancer site substring present in study title or any chunk |
| `histology` | Query histology substring present in study title / chunks |
| `biomarker` | Any query biomarker substring present in study title / chunks |
| `trajectory` | Any trajectory flag keyword (e.g. "refractory", "progression") present in study title / chunks |
| `source_lane` | `study.source` ∈ {postgres, pto, both} — indicates a structured / PTO match, which is higher-precision than qdrant-only |

In [ ]:
import re
from typing import Any, Dict, List

from src.api.services.comprehensive_retrieval import normalize_category

_CRITERION_WEIGHTS = {
    "category":    0.20,
    "site":        0.20,
    "histology":   0.15,
    "biomarker":   0.20,
    "trajectory":  0.15,
    "source_lane": 0.10,
}

_TRAJECTORY_KEYWORDS = {
    "ici_refractory":    ["ici-refractory", "anti-pd1", "checkpoint refract", "post-pembrolizumab", "post-nivolumab"],
    "progressing_on_ici":["progressing on ici", "progression on pembrolizumab", "progression on nivolumab"],
    "progression":       ["progression", "progressed", "progressing"],
    "tki_refractory":    ["tki-refractory", "osimertinib-resistant", "post-osimertinib", "egfr resistant"],
    "mCRPC":             ["castration-resistant", "mcrpc", "crpc"],
    "recurrence":        ["recurrence", "recurrent"],
}

def _study_blob(study) -> str:
    """Concatenate title + all chunk text for substring matching."""
    parts = [study.title or ""]
    for c in study.chunks or []:
        parts.append(c.get("text", "") or "")
    return " ".join(parts).lower()

def score_matching_criteria(study, query_axes: Dict[str, Any], filter_category: str | None) -> Dict[str, Any]:
    blob = _study_blob(study)
    matched: List[str] = []
    detail: Dict[str, Any] = {}

    # category — use normalize_category() so alias pairs (h&n / head_neck)
    # and _processed_documents suffixes are resolved before comparison.
    if filter_category and study.category and normalize_category(filter_category) == normalize_category(str(study.category)):
        matched.append("category")
        detail["category"] = f"{filter_category} ~= {study.category} (normalized)"

    # site
    site = query_axes.get("site")
    if site and site.lower() in blob:
        matched.append("site")
        detail["site"] = site

    # histology
    hist = query_axes.get("histology")
    if hist and hist.lower() in blob:
        matched.append("histology")
        detail["histology"] = hist

    # biomarkers
    bm_hits = [b for b in (query_axes.get("biomarkers") or []) if b.lower() in blob]
    if bm_hits:
        matched.append("biomarker")
        detail["biomarker"] = bm_hits

    # trajectory
    tr_hits = []
    for flag in (query_axes.get("trajectory") or []):
        for kw in _TRAJECTORY_KEYWORDS.get(flag, [flag.replace("_", " ")]):
            if kw in blob:
                tr_hits.append(flag)
                break
    if tr_hits:
        matched.append("trajectory")
        detail["trajectory"] = tr_hits

    # source lane
    if (study.source or "").lower() in {"postgres", "pto", "both"}:
        matched.append("source_lane")
        detail["source_lane"] = study.source

    criteria_score = sum(_CRITERION_WEIGHTS[c] for c in matched)
    return {
        "matched": matched,
        "detail": detail,
        "criteria_score": round(criteria_score, 3),
        "retrieval_score": round(float(study.score or 0), 3),
        "composite_score": round(criteria_score * 0.5 + min(float(study.score or 0), 1.0) * 0.5, 3),
    }

## 6. Per-query runner

Calls `retrieve_evidence(mode="comprehensive")`, invokes `pick_surface`,
and prints a structured report of:

- router decision (which surface the query would be sent to today)
- extracted axes the pipeline derived from the narrative
- top studies with retrieval score + criteria match breakdown
- active `PipelineMetrics` summary line

In [ ]:
import asyncio, time
from src.api.services.retrieval_backbone import retrieve_evidence
from src.api.services.unified_router import pick_surface
from src.api.services.query_structuring_service import structure_query_fast
from src.api.services import pipeline_metrics as _pm

async def run_case(q: Dict[str, Any], mode: str = "comprehensive", max_studies: int = 6):
    # Start a fresh metrics context so safety/source counters are isolated
    _pm.start("notebook-" + q["id"])

    # Structure the query → we use filter_category for criteria matching
    qs = structure_query_fast(q["narrative"], "treatment_recommendation")
    filter_category = getattr(qs, "filter_category", None)

    # Router
    axes_hint = {
        "has_patient_context": bool(getattr(qs, "has_patient_context", False)),
        "prior_treatments": list(getattr(getattr(qs, "treatment", None), "prior_treatments", []) or []),
        "trajectory_flags": q["expected_axes"].get("trajectory", []),
    }
    surface = pick_surface(axes_hint).value

    # Retrieval
    t0 = time.perf_counter()
    bundle = await retrieve_evidence(
        query_text=q["narrative"],
        mode=mode,
        max_studies=max_studies,
        chunks_per_study=6,
    )
    elapsed_ms = (time.perf_counter() - t0) * 1000

    # Score each returned study against the expected axes
    per_study = []
    for s in bundle.studies:
        per_study.append({
            "doc_id": s.doc_id,
            "title":  s.title,
            "category": s.category,
            "source":  s.source,
            "year":    s.year,
            "chunk_count": len(s.chunks or []),
            "sections": list(s.sections_covered or []),
            **score_matching_criteria(s, q["expected_axes"], filter_category),
        })
    per_study.sort(key=lambda r: r["composite_score"], reverse=True)

    report = {
        "query_id":        q["id"],
        "surface":         surface,
        "filter_category": filter_category,
        "extracted_axes":  bundle.extracted_axes,
        "elapsed_ms":      elapsed_ms,
        "studies":         per_study,
        "metrics_line":    _pm.current().summary_line() if _pm.current() else "",
        "source_provenance": dict(bundle.source_provenance),
    }
    return report

def print_report(rep: Dict[str, Any]) -> None:
    print("=" * 100)
    print(f"QUERY: {rep['query_id']}   |   surface={rep['surface']}   |   category={rep['filter_category']}   |   {rep['elapsed_ms']:.0f} ms")
    print("=" * 100)
    print("Extracted axes:")
    for k, v in (rep['extracted_axes'] or {}).items():
        print(f"  - {k}: {v}")
    print(f"\nSource provenance: {rep['source_provenance']}")
    print(f"\n{rep['metrics_line']}\n")
    print(f"{'rank':<4} {'comp':>5} {'crit':>5} {'retr':>5} {'src':<10} {'cat':<14} {'chunks':>6}  title")
    print("-" * 100)
    for i, s in enumerate(rep["studies"], start=1):
        print(f"{i:<4} {s['composite_score']:>5.2f} {s['criteria_score']:>5.2f} {s['retrieval_score']:>5.2f} "
              f"{(s['source'] or '')[:10]:<10} {(s['category'] or '')[:14]:<14} {s['chunk_count']:>6}  {s['title'][:70]}")
    print("\nPer-study matching criteria:")
    for i, s in enumerate(rep["studies"], start=1):
        print(f"  #{i} [{s['doc_id'][:32]}]")
        print(f"     matched: {s['matched'] or 'none'}")
        for k, v in (s['detail'] or {}).items():
            print(f"       · {k}: {v}")
        if s['sections']:
            print(f"     sections: {s['sections']}")

## 7. Run all three queries

In [ ]:
reports = []
for q in TEST_QUERIES:
    rep = await run_case(q, mode="comprehensive", max_studies=6)
    reports.append(rep)
    print_report(rep)
    print()

## 8. Side-by-side scoring summary

Aggregates per-query stats so you can see at a glance which cases the new
pipeline covered best. `mean_criteria` is the average of `criteria_score`
across the top 6 studies — higher means more of the extracted axes were
actually grounded in retrieved documents.

In [ ]:
from statistics import mean

print(f"{'query':<22} {'surface':<4} {'studies':>8} {'mean_crit':>10} {'mean_retr':>10} {'mean_comp':>10} {'ms':>6}")
print("-" * 80)
for r in reports:
    studies = r["studies"]
    if not studies:
        print(f"{r['query_id']:<22} {r['surface']:<4} {0:>8} {'—':>10} {'—':>10} {'—':>10} {r['elapsed_ms']:>6.0f}")
        continue
    mc = mean(s["criteria_score"]    for s in studies)
    mr = mean(s["retrieval_score"]   for s in studies)
    mp = mean(s["composite_score"]   for s in studies)
    print(f"{r['query_id']:<22} {r['surface']:<4} {len(studies):>8} {mc:>10.3f} {mr:>10.3f} {mp:>10.3f} {r['elapsed_ms']:>6.0f}")

# Criterion-level hit rate across all three queries
print("\nCriterion hit rate across all queries (fraction of studies matching):")
all_studies = [s for r in reports for s in r["studies"]]
if all_studies:
    for crit in ["category", "site", "histology", "biomarker", "trajectory", "source_lane"]:
        hit = sum(1 for s in all_studies if crit in s["matched"]) / len(all_studies)
        bar = "█" * int(hit * 30)
        print(f"  {crit:<12} {hit*100:>5.1f}%  {bar}")

## 9. Notes

- **Surface** here is always the *planned* surface from the shadow dispatcher
  (Phase 3). Today `/query/enhanced` always hits P1; the router logs its pick
  for observability but does not re-route traffic yet.
- `criteria_score` is a notebook-side heuristic, not produced by the pipeline
  itself. It validates that whatever the retriever returns actually mentions
  the axes the query structurer extracted — useful for spotting silent
  category leakage (Phase 1a bug class).
- `retrieval_score` is the pipeline's own rerank/cross-encoder score
  (`EvidenceStudy.score`).
- The `[PipelineMetrics] …` line comes from `pipeline_metrics.summary_line()`
  — look for `sources=`, `elig=`, `safety=` counters.
- Swap `mode="comprehensive"` for `"multispecialty"` or `"fast"` in cell 7
  to exercise the other backbones.
- To change queries, edit `TEST_QUERIES` in cell 4 and re-run cells 7 and 8.